# Building a GraphRAG Application with Amazon Bedrock Knowledge Bases and Amazon Neptune Analytics

## Introduction
Vector RAG finds the chunks that look most similar to a question. That works well for single facts, but it struggles when the answer depends on how several entities relate across a document, for example a supplier, a risk and a business segment.

**GraphRAG** adds a knowledge graph. While the documents are ingested, a foundation model extracts the entities and relationships in every chunk and stores them in a graph database. At question time the system runs a vector search first and then follows the graph to related chunks, so the model sees connected context instead of isolated passages.

This notebook builds the complete pipeline **from code only**. Every AWS resource is created, used and deleted with boto3:

| Component | Service or model | Role in this lab |
|---|---|---|
| Document store | **Amazon S3** | Holds the AnyCompany 10-K report |
| Graph database | **Amazon Neptune Analytics** | Stores the knowledge graph and the vector embeddings |
| GraphRAG engine | **Amazon Bedrock Knowledge Bases** | Chunks, embeds, extracts entities, builds the graph, retrieves |
| Embedding model | **Amazon Titan Text Embeddings V2** | Turns text into 1,024-number vectors |
| Graph construction model | **Amazon Nova Lite** | Extracts entities and relationships from every chunk |
| Answer and judge model | **Amazon Nova Lite** | Writes cited answers and checks them for hallucinations |

### What you will do
1. **Build the graph**: create the bucket, the graph, the service role, the Knowledge Base and the data source, then sync.
2. **Explore the graph**: query it directly with openCypher and rank entities with graph algorithms.
3. **Retrieve**: call the Retrieve API and build a vector-only baseline for comparison.
4. **Trust the answers**: generate answers with citations and rate their hallucination risk.
5. **Compare**: measure vector RAG against GraphRAG on fact, multi-hop and unanswerable questions.
6. **Go agentic**: let the model write its own openCypher queries.
7. **Clean up**: delete everything.

> **Cost warning:** the Neptune Analytics graph is billed for every hour it exists (about 0.48 USD per hour for the smallest size in us-east-1), from Step 5 until Step 23 deletes it. Deleting the Knowledge Base does **not** delete the graph. Always run Steps 23 and 24 at the end.

> **If the kernel restarts:** re-run Steps 1 to 3, then continue where you stopped. The file `lab_state.json` remembers every resource the notebook has created, so nothing is created twice.


# Setup

### 1. Installing Packages

This cell installs only the packages the notebook actually uses:

- **boto3** and **botocore**: the AWS SDK for Python. They are upgraded because Neptune Analytics and the GraphRAG entity extraction setting need a recent version.
- **pypdf**: reads the text of the PDF report for the vector-only baseline.
- **requests**: downloads the report from GitHub.
- **numpy** and **pandas**: vector maths and readable tables.

**Note:** if this cell upgrades boto3, restart the kernel (Kernel menu, Restart Kernel) and continue from Step 2.


In [ ]:
%pip install --quiet --upgrade boto3 botocore
%pip install --quiet pypdf requests numpy pandas

print('All packages installed successfully!')

### 2. Configuring the AWS Session and Lab Settings

- `REGION = 'us-east-1'`: every resource is created in US East (N. Virginia), where Bedrock GraphRAG is available.
- **The state file**: `lab_state.json` stores the ID of every resource as soon as it is created. `save_state()` writes to it. This is what makes the cleanup cell reliable.
- **Resource names**: a random 6-character `SUFFIX` makes the names unique, so several learners can use the same account.
- **Models**:
  - `EMBED_MODEL_ID`: Titan Text Embeddings V2 with 1,024 dimensions. The graph must be created with the same dimension.
  - `GRAPH_MODEL_ARN`: the graph construction model that extracts entities and relationships. A commented alternative uses Claude Haiku 4.5.
  - `GEN_MODEL_ID`: the model that writes and judges the answers.
- **Clients**: one boto3 client per service. `neptune-graph` is the client for Neptune Analytics. `bedrock-agent` creates Knowledge Bases, and `bedrock-agent-runtime` queries them.
- `wait_for()`: a helper that checks a status every few seconds and prints it only when it changes.


In [ ]:
import os, json, time, uuid, re, random
import boto3, botocore, requests
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from botocore.config import Config
from botocore.exceptions import ClientError

REGION = 'us-east-1'
session = boto3.Session(region_name=REGION)
ACCOUNT_ID = session.client('sts').get_caller_identity()['Account']

# The state file remembers every resource this notebook creates
STATE_FILE = 'lab_state.json'
state = json.load(open(STATE_FILE)) if os.path.exists(STATE_FILE) else {}

def save_state(**values):
    state.update(values)
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2)

SUFFIX = state.get('suffix') or uuid.uuid4().hex[:6]
save_state(suffix=SUFFIX)

BUCKET_NAME = 'k21-graphrag-' + ACCOUNT_ID + '-' + SUFFIX
ROLE_NAME = 'K21GraphRAGKBRole-' + SUFFIX
POLICY_NAME = 'K21GraphRAGKBPolicy'
GRAPH_NAME = 'k21-graphrag-' + SUFFIX
KB_NAME = 'k21-graphrag-kb-' + SUFFIX
DS_NAME = 'k21-graphrag-ds-' + SUFFIX

EMBED_MODEL_ID = 'amazon.titan-embed-text-v2:0'
EMBED_DIM = 1024
EMBED_MODEL_ARN = 'arn:aws:bedrock:' + REGION + '::foundation-model/' + EMBED_MODEL_ID

# Model that extracts entities and relationships while the graph is built
GRAPH_MODEL_ARN = 'arn:aws:bedrock:' + REGION + '::foundation-model/amazon.nova-lite-v1:0'
# Alternative: Claude Haiku 4.5 through a US cross-Region inference profile
# GRAPH_MODEL_ARN = 'arn:aws:bedrock:' + REGION + ':' + ACCOUNT_ID + ':inference-profile/us.anthropic.claude-haiku-4-5-20251001-v1:0'

# Model that writes the answers and judges them
GEN_MODEL_ID = 'amazon.nova-lite-v1:0'

cfg = Config(connect_timeout=10, read_timeout=300, retries={'max_attempts': 5, 'mode': 'adaptive'})
s3 = session.client('s3')
iam = session.client('iam')
neptune = session.client('neptune-graph', config=cfg)
bedrock_agent = session.client('bedrock-agent')
bedrock_agent_runtime = session.client('bedrock-agent-runtime', config=cfg)
bedrock_runtime = session.client('bedrock-runtime', config=cfg)

def wait_for(get_status, done, failed=(), every=20, timeout=3600, label=''):
    start, last = time.time(), None
    while True:
        status = get_status()
        if status != last:
            print('[' + label + ']', status, '(' + str(int(time.time() - start)) + 's)')
            last = status
        if status in done:
            return status
        if status in failed:
            raise RuntimeError(label + ' ended in status ' + status)
        if time.time() - start > timeout:
            raise TimeoutError(label)
        time.sleep(every)

print('Account:', ACCOUNT_ID)
print('Region:', REGION)
print('Suffix:', SUFFIX)
print('Bucket:', BUCKET_NAME)
print('Role:', ROLE_NAME)
print('Graph:', GRAPH_NAME)
print('Knowledge Base:', KB_NAME)

### 3. Checking Library Versions and Permissions

Before creating anything that costs money, this cell checks that everything is in place:

- **Library versions**: the Neptune Analytics client must have `create_graph` and `execute_query`.
- **GraphRAG parameter**: the Bedrock client must know the `contextEnrichmentConfiguration` setting, which is how entity extraction is switched on. The cell reads it from the SDK's own service model.
- **Permissions**: a harmless list call to Neptune Analytics and to Bedrock Knowledge Bases proves that the SageMaker execution role can reach both services.

**Expected output:** a series of `OK` lines ending with `Ready to build the lab.` If you see `MISSING`, follow the message printed under it.


In [ ]:
def report(ok, name):
    print(('OK       ' if ok else 'MISSING  ') + name)
    return ok

print('boto3 version:', boto3.__version__)
print('botocore version:', botocore.__version__)
print()

results = []
results.append(report(hasattr(neptune, 'create_graph'), 'Neptune Analytics client: create_graph'))
results.append(report(hasattr(neptune, 'execute_query'), 'Neptune Analytics client: execute_query'))
results.append(report(hasattr(bedrock_agent, 'create_knowledge_base'), 'Bedrock Agent client: create_knowledge_base'))

try:
    shape = bedrock_agent.meta.service_model.shape_for('VectorIngestionConfiguration')
    has_enrichment = 'contextEnrichmentConfiguration' in shape.members
except Exception:
    has_enrichment = False
results.append(report(has_enrichment, 'GraphRAG entity extraction parameter'))

try:
    neptune.list_graphs(maxResults=5)
    results.append(report(True, 'Permissions: Neptune Analytics'))
except ClientError as e:
    results.append(report(False, 'Permissions: Neptune Analytics (' + e.response['Error']['Code'] + ')'))

try:
    bedrock_agent.list_knowledge_bases(maxResults=5)
    results.append(report(True, 'Permissions: Bedrock Knowledge Bases'))
except ClientError as e:
    results.append(report(False, 'Permissions: Bedrock Knowledge Bases (' + e.response['Error']['Code'] + ')'))

print()
if all(results):
    print('Ready to build the lab.')
else:
    print('Something is missing.')
    print('- Library problem: restart the kernel and run again from Step 2.')
    print('- Permission problem: add the inline policy from section 6.2 of the lab guide to the SageMaker execution role.')

# Building the Knowledge Graph

### 4. Creating the S3 Bucket and Uploading the Report

- `requests.get(DATA_URL)` downloads AnyCompany's annual financial report (10-K) into a local `docs/` folder. The first four bytes are checked to make sure the file really is a PDF.
- `s3.create_bucket()` creates the bucket. In us-east-1 no location setting is needed.
- Every supported file in `docs/` is uploaded under the `source/` prefix. The data source in Step 8 reads only that prefix.

**Try it:** before running this cell, copy extra related documents (PDF, TXT, MD, DOCX, HTML or CSV) into the `docs/` folder. GraphRAG shows its value best when it can connect several documents.


In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/k21academyuk/AWS-GenAI_AIML/main/AnyCompany_financial_10K.pdf'
os.makedirs('docs', exist_ok=True)
local_pdf = 'docs/AnyCompany_financial_10K.pdf'

if not os.path.exists(local_pdf):
    response = requests.get(DATA_URL, timeout=120)
    response.raise_for_status()
    if response.content[:4] != b'%PDF':
        raise ValueError('The download is not a PDF. Check DATA_URL.')
    with open(local_pdf, 'wb') as f:
        f.write(response.content)
print('Report ready:', local_pdf, '(' + str(os.path.getsize(local_pdf) // 1024) + ' KB)')

if 'bucket' not in state:
    s3.create_bucket(Bucket=BUCKET_NAME)
    save_state(bucket=BUCKET_NAME)

ALLOWED = ('.pdf', '.txt', '.md', '.docx', '.html', '.csv')
for name in sorted(os.listdir('docs')):
    if name.lower().endswith(ALLOWED):
        s3.upload_file(os.path.join('docs', name), BUCKET_NAME, 'source/' + name)
        print('Uploaded: s3://' + BUCKET_NAME + '/source/' + name)

### 5. Creating the Amazon Neptune Analytics Graph

Neptune Analytics is the in-memory graph engine that Bedrock GraphRAG uses. It stores both the graph and the vector embeddings.

- `provisionedMemory=16`: the graph size in memory-optimized Neptune Capacity Units (m-NCU). 16 is the smallest size.
- `replicaCount=0`: no standby copy. Fine for a lab, not for production.
- `vectorSearchConfiguration={'dimension': 1024}`: turns on vector search. The dimension **must** match the embedding model.
- `publicConnectivity=True`: lets this notebook reach the graph without VPC setup. Every request is still signed with IAM, so the graph is not open to the internet. Production systems use a private endpoint.
- `deletionProtection=False`: allows the cleanup cell to delete the graph.

The call returns immediately and the graph is created in the background, which takes about 10 to 15 minutes. Step 7 waits for it.

**Billing starts now.**


In [ ]:
if 'graph_id' not in state:
    response = neptune.create_graph(
        graphName=GRAPH_NAME,
        provisionedMemory=16,
        replicaCount=0,
        publicConnectivity=True,
        deletionProtection=False,
        vectorSearchConfiguration={'dimension': EMBED_DIM},
        tags={'project': 'k21-graphrag-lab'},
    )
    save_state(graph_id=response['id'], graph_arn=response['arn'])

GRAPH_ID = state['graph_id']
GRAPH_ARN = state['graph_arn']
print('Graph ID:', GRAPH_ID)
print('Graph ARN:', GRAPH_ARN)
print('Status:', neptune.get_graph(graphIdentifier=GRAPH_ID)['status'])
print('Billing has started. Remember to run Step 23 at the end of the lab.')

### 6. Creating the Knowledge Base Service Role

Bedrock needs its own IAM role to work on your behalf. You can run this cell while the graph is still being created.

- **Trust policy**: allows only `bedrock.amazonaws.com` to assume the role, and only for Knowledge Bases in **your** account and Region. The two conditions prevent another account from using your role.
- **Permissions policy**, four statements:
  - `BedrockModels`: call the embedding model and the graph construction model, including through inference profiles.
  - `S3List` and `S3Read`: list and read the lab bucket only.
  - `NeptuneAnalytics`: read and write **this** graph only.
- `iam.put_role_policy()` attaches the permissions as an inline policy.
- The 15-second pause gives IAM time to make the new role visible to Bedrock.


In [ ]:
trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'bedrock.amazonaws.com'},
        'Action': 'sts:AssumeRole',
        'Condition': {
            'StringEquals': {'aws:SourceAccount': ACCOUNT_ID},
            'ArnLike': {'aws:SourceArn': 'arn:aws:bedrock:' + REGION + ':' + ACCOUNT_ID + ':knowledge-base/*'},
        },
    }],
}

permissions_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {'Sid': 'BedrockModels', 'Effect': 'Allow',
         'Action': ['bedrock:InvokeModel', 'bedrock:GetInferenceProfile'],
         'Resource': ['arn:aws:bedrock:*::foundation-model/*',
                      'arn:aws:bedrock:*:' + ACCOUNT_ID + ':inference-profile/*']},
        {'Sid': 'S3List', 'Effect': 'Allow', 'Action': ['s3:ListBucket'],
         'Resource': ['arn:aws:s3:::' + BUCKET_NAME]},
        {'Sid': 'S3Read', 'Effect': 'Allow', 'Action': ['s3:GetObject'],
         'Resource': ['arn:aws:s3:::' + BUCKET_NAME + '/*']},
        {'Sid': 'NeptuneAnalytics', 'Effect': 'Allow', 'Action': ['neptune-graph:*'],
         'Resource': [GRAPH_ARN]},
    ],
}

if 'role_arn' not in state:
    role = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='K21 GraphRAG lab - Bedrock Knowledge Base service role',
    )
    save_state(role_name=ROLE_NAME, role_arn=role['Role']['Arn'])

iam.put_role_policy(RoleName=ROLE_NAME, PolicyName=POLICY_NAME, PolicyDocument=json.dumps(permissions_policy))
ROLE_ARN = state['role_arn']
print('Role ARN:', ROLE_ARN)
print('Waiting 15 seconds for IAM to propagate...')
time.sleep(15)
print('Done.')

### 7. Creating the GraphRAG Knowledge Base

- `wait_for(...)` first waits until the graph status is `AVAILABLE`.
- `create_knowledge_base()` then creates the Knowledge Base:
  - `knowledgeBaseConfiguration`: a vector Knowledge Base using Titan Text Embeddings V2, 1,024 dimensions, 32-bit floats.
  - `storageConfiguration`: type `NEPTUNE_ANALYTICS`. This single choice is what makes it a GraphRAG Knowledge Base. `fieldMapping` names the fields where the chunk text and metadata are stored.
- The `for attempt` loop retries for about 90 seconds if Bedrock cannot see the new IAM role yet. This is normal right after a role is created.
- The last `wait_for` waits until the Knowledge Base is `ACTIVE`.

**Expected output:** status lines for the graph and the Knowledge Base, then the Knowledge Base ID.


In [ ]:
wait_for(lambda: neptune.get_graph(graphIdentifier=GRAPH_ID)['status'],
         done=('AVAILABLE',), failed=('FAILED',), every=30, label='Neptune graph')

if 'kb_id' not in state:
    for attempt in range(6):
        try:
            kb = bedrock_agent.create_knowledge_base(
                name=KB_NAME,
                description='K21 GraphRAG lab - AnyCompany financial report',
                roleArn=ROLE_ARN,
                knowledgeBaseConfiguration={
                    'type': 'VECTOR',
                    'vectorKnowledgeBaseConfiguration': {
                        'embeddingModelArn': EMBED_MODEL_ARN,
                        'embeddingModelConfiguration': {
                            'bedrockEmbeddingModelConfiguration': {
                                'dimensions': EMBED_DIM,
                                'embeddingDataType': 'FLOAT32',
                            }
                        },
                    },
                },
                storageConfiguration={
                    'type': 'NEPTUNE_ANALYTICS',
                    'neptuneAnalyticsConfiguration': {
                        'graphArn': GRAPH_ARN,
                        'fieldMapping': {'textField': 'text', 'metadataField': 'metadata'},
                    },
                },
            )['knowledgeBase']
            save_state(kb_id=kb['knowledgeBaseId'])
            break
        except ClientError as e:
            code = e.response['Error']['Code']
            if code in ('ValidationException', 'AccessDeniedException') and attempt < 5:
                print('Waiting for the IAM role to propagate:', e.response['Error']['Message'][:120])
                time.sleep(15)
            else:
                raise

KB_ID = state['kb_id']
wait_for(lambda: bedrock_agent.get_knowledge_base(knowledgeBaseId=KB_ID)['knowledgeBase']['status'],
         done=('ACTIVE',), failed=('FAILED',), every=10, label='Knowledge Base')
print('Knowledge Base ID:', KB_ID)

### 8. Creating the Data Source with Entity Extraction

The data source tells the Knowledge Base where the documents are and how to process them.

- `dataSourceConfiguration`: the lab bucket, limited to the `source/` prefix.
- `chunkingConfiguration`: fixed-size chunks of 300 tokens with 20 percent overlap, the same as the previous lab, so results are comparable.
- `contextEnrichmentConfiguration`: **the GraphRAG part.** The method `CHUNK_ENTITY_EXTRACTION` sends every chunk to the graph construction model, which returns the entities and relationships it finds. Bedrock writes them into the graph and links them to the chunks they came from.
- `dataDeletionPolicy='RETAIN'`: when the data source is deleted, its data is left in the graph. Because the cleanup deletes the whole graph anyway, this makes the cleanup faster and avoids failures.


In [ ]:
if 'ds_id' not in state:
    ds = bedrock_agent.create_data_source(
        knowledgeBaseId=KB_ID,
        name=DS_NAME,
        dataDeletionPolicy='RETAIN',
        dataSourceConfiguration={
            'type': 'S3',
            's3Configuration': {
                'bucketArn': 'arn:aws:s3:::' + BUCKET_NAME,
                'inclusionPrefixes': ['source/'],
            },
        },
        vectorIngestionConfiguration={
            'chunkingConfiguration': {
                'chunkingStrategy': 'FIXED_SIZE',
                'fixedSizeChunkingConfiguration': {'maxTokens': 300, 'overlapPercentage': 20},
            },
            'contextEnrichmentConfiguration': {
                'type': 'BEDROCK_FOUNDATION_MODEL',
                'bedrockFoundationModelConfiguration': {
                    'modelArn': GRAPH_MODEL_ARN,
                    'enrichmentStrategyConfiguration': {'method': 'CHUNK_ENTITY_EXTRACTION'},
                },
            },
        },
    )['dataSource']
    save_state(ds_id=ds['dataSourceId'])

DS_ID = state['ds_id']
print('Data source ID:', DS_ID)
print('Graph construction model:', GRAPH_MODEL_ARN.split('/')[-1])

### 9. Syncing the Data Source to Build the Graph

`start_ingestion_job()` starts the sync. For every document, Bedrock:

1. splits it into chunks,
2. turns each chunk into a vector with Titan,
3. asks the graph construction model for the entities and relationships in the chunk,
4. writes chunks, vectors, entities and relationships into the Neptune Analytics graph.

The cell waits until the job is `COMPLETE`, then prints the statistics. This takes about 10 to 30 minutes, depending on the number of documents.

**What to look for:** `numberOfDocumentsFailed` should be 0. If the job fails, the failure reasons are printed. See the Troubleshooting section of the lab guide.


In [ ]:
job = bedrock_agent.start_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)['ingestionJob']
JOB_ID = job['ingestionJobId']
save_state(last_job_id=JOB_ID)

def get_job():
    return bedrock_agent.get_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID,
                                           ingestionJobId=JOB_ID)['ingestionJob']

status = wait_for(lambda: get_job()['status'], done=('COMPLETE', 'FAILED', 'STOPPED'),
                  every=30, timeout=7200, label='Sync: chunk, embed, extract entities, build graph')

final_job = get_job()
print(json.dumps(final_job.get('statistics', {}), indent=2))
if final_job.get('failureReasons'):
    print('Failure reasons:', final_job['failureReasons'])
if status != 'COMPLETE':
    raise RuntimeError('The sync did not complete. Check the failure reasons above.')

# Exploring the Knowledge Graph

### 10. Connecting to the Graph with openCypher

Bedrock built the graph for you. Now you can look inside it directly.

- `cypher(query, params)`: a small helper that sends an **openCypher** query to the graph with `execute_query()` and returns the result rows. `params` passes values safely, the same way SQL parameters do.
- `short()`: shortens long values so tables stay readable.
- The first query counts nodes per **label** (the type of a node). `UNWIND` turns each node's list of labels into separate rows.
- The second query counts edges per **relationship type**. The pattern `()-[r]->()` means any node, an edge `r`, any node.

**What to look for:** note the label that holds the extracted entities. Step 12 uses it.


In [ ]:
def cypher(query, params=None):
    request = {'graphIdentifier': GRAPH_ID, 'queryString': query, 'language': 'OPEN_CYPHER'}
    if params:
        request['parameters'] = params
    response = neptune.execute_query(**request)
    return json.loads(response['payload'].read())['results']

def short(value, n=80):
    text = value if isinstance(value, str) else json.dumps(value, default=str)
    return text if len(text) <= n else text[:n] + '...'

labels = cypher('MATCH (n) UNWIND labels(n) AS label RETURN label, count(*) AS nodes ORDER BY nodes DESC')
rels = cypher('MATCH ()-[r]->() RETURN type(r) AS relationship, count(*) AS edges ORDER BY edges DESC')

if not labels:
    print('The graph is empty. Check that Step 9 finished with COMPLETE.')
else:
    print('Node labels')
    display(pd.DataFrame(labels))
    print('Relationship types')
    display(pd.DataFrame(rels))

### 11. Looking Inside the Nodes

- For every label, `properties(n)` returns all properties of two sample nodes as a dictionary.
- Label names are wrapped in backticks, because some labels contain characters such as underscores that openCypher would otherwise misread.

**What to look for:** which property holds the entity name, and which holds the chunk text. Bedrock decides the property names, which is why the next cell detects them instead of hard-coding them.


In [ ]:
for row in labels:
    label = row['label']
    samples = cypher('MATCH (n:`' + label + '`) RETURN properties(n) AS props LIMIT 2')
    print('===', label, '===')
    for s in samples:
        print({k: short(v) for k, v in s['props'].items()})
    print()

### 12. Finding the Most Connected Entities

- `ENTITY_LABEL`: the first label whose name contains entity. If none does, the most common label is used.
- `NAME_PROP`: the first property that looks like a name.
- `name_of(var)`: builds the openCypher expression that shows a readable name for a node, falling back to its ID.
- The query counts the relationships of every entity, its **degree**, and keeps the top 15.

**What to look for:** high-degree entities are **hubs**. They connect many parts of the report, which is exactly what GraphRAG uses to jump from one chunk to related chunks. Expect the company itself and its major themes near the top.


In [ ]:
ENTITY_LABEL = next((r['label'] for r in labels if 'entity' in r['label'].lower()), labels[0]['label'])
sample_rows = cypher('MATCH (n:`' + ENTITY_LABEL + '`) RETURN properties(n) AS props LIMIT 20')
entity_keys = sorted({k for r in sample_rows for k in r['props'].keys()})
NAME_PROP = next((k for k in ['name', 'entity_name', 'entity', 'value', 'title', 'text'] if k in entity_keys), None)

def name_of(var):
    if NAME_PROP:
        return 'coalesce(' + var + '.`' + NAME_PROP + '`, id(' + var + '))'
    return 'id(' + var + ')'

print('Entity label:', ENTITY_LABEL)
print('Entity properties:', entity_keys)
print('Name property used:', NAME_PROP)

hubs = cypher(
    'MATCH (n:`' + ENTITY_LABEL + '`)-[r]-() '
    'RETURN ' + name_of('n') + ' AS entity, count(r) AS degree '
    'ORDER BY degree DESC LIMIT 15'
)
display(pd.DataFrame([{k: short(v) for k, v in h.items()} for h in hubs]))

### 13. Exploring an Entity's Neighbourhood

- `TOP_ENTITY` is the most connected entity from the previous step.
- The pattern `(n)-[r]-(m)` matches every edge in either direction between the entity `n` and a neighbour `m`.
- `$name` is a **query parameter**. The value is sent separately from the query text, which avoids quoting problems and query injection.

**Try it:** set `TOP_ENTITY` to another name from the table above, for example a supplier or a risk, and run the cell again. This is the kind of hop GraphRAG makes automatically during retrieval.


In [ ]:
TOP_ENTITY = hubs[0]['entity']
neighbours = cypher(
    'MATCH (n:`' + ENTITY_LABEL + '`)-[r]-(m) '
    'WHERE ' + name_of('n') + ' = $name '
    'RETURN type(r) AS relationship, labels(m) AS neighbour_type, ' + name_of('m') + ' AS neighbour '
    'LIMIT 25',
    {'name': TOP_ENTITY},
)
print('Neighbourhood of:', TOP_ENTITY)
if neighbours:
    display(pd.DataFrame([{k: short(v) for k, v in r.items()} for r in neighbours]))
else:
    print('No neighbours found.')

### 14. Ranking Entities with PageRank

Degree counts connections. **PageRank** goes one step further: an entity scores higher when it is connected to other important entities, not just to many entities. It is the algorithm Google originally used to rank web pages.

- `CALL neptune.algo.pageRank(...)` runs the algorithm inside Neptune Analytics, as part of an openCypher query. No data leaves the graph.
- `numOfIterations: 20` sets how many rounds the scores are refined.
- If your graph does not support this call syntax, the cell prints a message and shows the degree ranking instead, so the lab continues.

**What to look for:** compare the PageRank list with the degree list. Entities that rise are connected to the important hubs. Entities that fall have many connections to minor entities.


In [ ]:
try:
    ranked = cypher(
        'MATCH (n:`' + ENTITY_LABEL + '`) '
        'CALL neptune.algo.pageRank(n, {numOfIterations: 20}) '
        'YIELD rank '
        'RETURN ' + name_of('n') + ' AS entity, rank ORDER BY rank DESC LIMIT 10'
    )
    print('PageRank: entities connected to other important entities score higher')
    display(pd.DataFrame([{k: short(v) for k, v in r.items()} for r in ranked]))
except ClientError as e:
    print('PageRank is not available on this graph:', e.response['Error']['Message'][:200])
    print('Showing the degree ranking from Step 12 instead.')
    display(pd.DataFrame([{k: short(v) for k, v in h.items()} for h in hubs[:10]]))

# GraphRAG Retrieval

### 15. Retrieving Context from the GraphRAG Knowledge Base

The Retrieve API returns chunks without generating an answer, exactly like the previous lab. On a Neptune Analytics Knowledge Base, the call does more work inside:

1. a vector search finds the most similar chunks,
2. the graph is followed from those chunks to the entities they mention,
3. chunks linked to the same entities are added to the result.

The helper `graph_retrieve()`:
- calls `retrieve()` with `numberOfResults=8`,
- labels each chunk `S1`, `S2` and so on for citations,
- keeps the text, score, file name, page and S3 URI,
- measures the latency.


In [ ]:
def to_page(value):
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None

def graph_retrieve(query, k=8):
    start = time.time()
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={'text': query},
        retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': k}},
    )
    chunks = []
    for i, result in enumerate(response['retrievalResults'], 1):
        metadata = result.get('metadata') or {}
        uri = (result.get('location', {}).get('s3Location', {}).get('uri')
               or metadata.get('x-amz-bedrock-kb-source-uri', ''))
        chunks.append({
            'id': 'S' + str(i),
            'text': result['content']['text'],
            'score': result.get('score'),
            'file': uri.split('/')[-1],
            'page': to_page(metadata.get('x-amz-bedrock-kb-document-page-number')),
            'uri': uri,
        })
    return chunks, time.time() - start

DEMO_QUESTION = 'What are the lease liabilities and the remaining lease terms?'
chunks, latency = graph_retrieve(DEMO_QUESTION)
print('Question:', DEMO_QUESTION)
print(len(chunks), 'chunks retrieved in', round(latency, 2), 'seconds')
print()
for c in chunks[:3]:
    print('[' + c['id'] + ']', c['file'], 'page', c['page'], '| score', c['score'])
    print(c['text'][:400])
    print('-' * 70)

### 16. Building a Vector-Only Baseline

To know whether the graph helps, you need something to compare it with. This cell builds a plain vector RAG pipeline **inside the notebook**, with no extra AWS resource.

- `PdfReader` extracts the text of each page.
- `chunk_words()` splits it into chunks of 220 words (about 300 tokens) with 20 percent overlap, close to the Knowledge Base settings.
- `embed()` turns each chunk into a vector with the same Titan model. `ThreadPoolExecutor` sends 8 requests at a time to save time.
- `vector_retrieve()` embeds the question and ranks chunks by **cosine similarity**. The vectors are normalized, so a matrix product gives the similarity directly.

This takes one to three minutes, depending on the size of the report.

**What to look for:** compare the pages retrieved here with the pages retrieved in Step 15 for the same question.


In [ ]:
from pypdf import PdfReader

def embed(text):
    body = json.dumps({'inputText': text[:8000], 'dimensions': EMBED_DIM, 'normalize': True})
    response = bedrock_runtime.invoke_model(modelId=EMBED_MODEL_ID, body=body)
    return json.loads(response['body'].read())['embedding']

def chunk_words(text, size=220, overlap=44):
    words = text.split()
    step = size - overlap
    return [' '.join(words[i:i + size]) for i in range(0, len(words), step) if words[i:i + size]]

vector_chunks = []
for name in sorted(os.listdir('docs')):
    if name.lower().endswith('.pdf'):
        for page_number, page in enumerate(PdfReader(os.path.join('docs', name)).pages, 1):
            for piece in chunk_words(page.extract_text() or ''):
                vector_chunks.append({'file': name, 'page': page_number, 'text': piece,
                                      'uri': 's3://' + BUCKET_NAME + '/source/' + name})

print('Embedding', len(vector_chunks), 'chunks with Titan Text Embeddings V2...')
with ThreadPoolExecutor(max_workers=8) as pool:
    VECTOR_MATRIX = np.array(list(pool.map(lambda c: embed(c['text']), vector_chunks)))
print('Vector baseline index:', len(vector_chunks), 'chunks')

def vector_retrieve(query, k=8):
    start = time.time()
    scores = VECTOR_MATRIX @ np.array(embed(query))
    chunks = []
    for i, idx in enumerate(np.argsort(-scores)[:k], 1):
        c = vector_chunks[idx]
        chunks.append({'id': 'S' + str(i), 'text': c['text'], 'score': round(float(scores[idx]), 4),
                       'file': c['file'], 'page': c['page'], 'uri': c['uri']})
    return chunks, time.time() - start

chunks, latency = vector_retrieve(DEMO_QUESTION)
print()
print('Vector-only retrieval for the same question:', round(latency, 2), 'seconds')
for c in chunks[:3]:
    print('[' + c['id'] + ']', c['file'], 'page', c['page'], '| score', c['score'])
    print(c['text'][:400])
    print('-' * 70)

# Trustworthy Answers

### 17. Generating Answers with Citations

- `log_event()`: writes one JSON line per event to `logs/graphrag_log.jsonl`, so every answer can be traced back to the chunks it was based on.
- `llm()`: calls Amazon Nova Lite with the **Converse API**. `temperature=0` makes answers repeatable. If the account requires an inference profile for this model, it retries automatically with the `us.` profile.
- `ANSWER_SYSTEM`: the rules for the model. Answer only from the numbered sources, cite a source after every fact, connect facts across sources when needed, and say so when the report does not contain the answer.
- `generate()`: puts each chunk in the prompt with its label, file and page.
- `references()`: lists the chunks that the answer actually cited.

**What to look for:** every sentence ends with a citation such as [S1], and each citation maps to a file, page and S3 URI.


In [ ]:
os.makedirs('logs', exist_ok=True)
LOG_FILE = 'logs/graphrag_log.jsonl'

def log_event(**event):
    event['time'] = time.strftime('%Y-%m-%dT%H:%M:%S')
    with open(LOG_FILE, 'a') as f:
        f.write(json.dumps(event, default=str) + '\n')

def llm(prompt, system=None, max_tokens=800):
    request = {
        'messages': [{'role': 'user', 'content': [{'text': prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': 0},
    }
    if system:
        request['system'] = [{'text': system}]
    last_error = None
    for model_id in (GEN_MODEL_ID, 'us.' + GEN_MODEL_ID):
        try:
            response = bedrock_runtime.converse(modelId=model_id, **request)
            return response['output']['message']['content'][0]['text'], response['usage']
        except ClientError as e:
            last_error = e
            message = str(e).lower()
            if 'on-demand' not in message and 'inference profile' not in message:
                raise
    raise last_error

ANSWER_SYSTEM = (
    'You are a financial analyst assistant. Answer ONLY from the numbered sources. '
    'Put a citation such as [S1] after every factual sentence. '
    'When the question needs it, connect facts from different sources and cite each one. '
    'If the sources do not contain the answer, reply exactly: The report does not contain this information. '
    'Never use outside knowledge.'
)

def generate(question, chunks):
    context = '\n\n'.join(
        '[' + c['id'] + '] (file: ' + c['file'] + ', page: ' + str(c['page']) + ')\n' + c['text'] for c in chunks
    )
    return llm('Sources:\n' + context + '\n\nQuestion: ' + question, system=ANSWER_SYSTEM)

def references(answer, chunks):
    return [c for c in chunks if '[' + c['id'] + ']' in answer]

chunks, _ = graph_retrieve(DEMO_QUESTION)
answer, usage = generate(DEMO_QUESTION, chunks)
print('Question:', DEMO_QUESTION)
print()
print(answer)
print()
print('References')
for c in references(answer, chunks):
    print('[' + c['id'] + ']', c['file'], 'page', c['page'], '-', c['uri'])
print()
print('Input tokens:', usage['inputTokens'], '| Output tokens:', usage['outputTokens'])

### 18. Detecting Hallucinations

An answer hallucinates when it states something that is not in the sources. Three checks are combined:

- **Citation check**: every factual sentence must cite a source, and the source must exist.
- **Number check**: every number in a sentence must appear in the source it cites. Numbers are the most damaging thing to get wrong in a financial answer.
- **LLM judge**: `judge_sentences()` asks a second model call to mark each sentence SUPPORTED or NOT_SUPPORTED by the sources.

`check_answer()` adds up the problems and gives a risk level:

| Problems found | Risk level |
|---|---|
| 0 | 🟢 LOW |
| 1 | 🟡 MEDIUM |
| 2 or more | 🔴 HIGH |
| The model said the report does not contain the answer | ⚪ ABSTAINED |


In [ ]:
CITATION = re.compile(r'\[(S\d+)\]')
NUMBER = re.compile(r'\d[\d,]*\.?\d*')
RISK_ICON = {'LOW': '🟢', 'MEDIUM': '🟡', 'HIGH': '🔴', 'ABSTAINED': '⚪'}

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if len(s.split()) > 3]

def judge_sentences(sentences, chunks):
    context = '\n\n'.join('[' + c['id'] + '] ' + c['text'] for c in chunks)
    prompt = ('Sources:\n' + context + '\n\nStatements:\n' + '\n'.join('- ' + s for s in sentences) +
              '\n\nFor each statement decide whether the sources support it. '
              'Return only a JSON list of objects with the keys sentence and verdict, '
              'where verdict is SUPPORTED or NOT_SUPPORTED.')
    text, _ = llm(prompt, max_tokens=1500)
    match = re.search(r'\[.*\]', text, re.S)
    try:
        verdicts = json.loads(match.group(0)) if match else []
    except json.JSONDecodeError:
        verdicts = []
    return [v for v in verdicts if isinstance(v, dict)]

def check_answer(answer, chunks):
    if 'does not contain this information' in answer.lower():
        return {'risk': 'ABSTAINED', 'issues': [], 'not_supported': 0}
    sources = {c['id']: c['text'].replace(',', '') for c in chunks}
    sentences = split_sentences(answer)
    issues = []
    for s in sentences:
        cited = CITATION.findall(s)
        if not cited:
            issues.append(('NO_CITATION', s))
            continue
        if any(c not in sources for c in cited):
            issues.append(('INVALID_CITATION', s))
            continue
        cited_text = ' '.join(sources[c] for c in cited)
        numbers = NUMBER.findall(CITATION.sub('', s))
        missing = [n for n in numbers if n.replace(',', '').rstrip('.') not in cited_text]
        if missing:
            issues.append(('NUMBER_NOT_IN_SOURCE: ' + ', '.join(missing), s))
    verdicts = judge_sentences(sentences, chunks) if sentences else []
    not_supported = sum(1 for v in verdicts if str(v.get('verdict', '')).upper() == 'NOT_SUPPORTED')
    total = len(issues) + not_supported
    risk = 'LOW' if total == 0 else ('MEDIUM' if total == 1 else 'HIGH')
    return {'risk': risk, 'issues': issues, 'not_supported': not_supported}

result = check_answer(answer, chunks)
print('Risk level:', RISK_ICON[result['risk']], result['risk'])
print('Statements the judge marked NOT_SUPPORTED:', result['not_supported'])
for kind, sentence in result['issues']:
    print('-', kind, '|', sentence[:150])

# Vector RAG vs GraphRAG

### 19. Comparing Vector RAG and GraphRAG

This is the main experiment of the lab. Five questions of three types go through both pipelines:

- **fact**: the answer is stated in one place. Vector RAG should do well.
- **multi-hop**: the answer needs facts from several sections connected together. This is where GraphRAG should help.
- **unanswerable**: the report does not contain the answer. Both pipelines should abstain.

For every question and pipeline, the cell retrieves, generates a cited answer, checks it for hallucinations and logs everything. Then `pairwise_judge()` asks the model which of the two answers is better. The order of the two answers is shuffled, because LLM judges tend to prefer whichever answer they read first.

**What to look for:**
- Who wins on the multi-hop questions, and who wins on the fact question?
- How many more input tokens does GraphRAG send to the model? Tokens are what you pay for.
- Do both pipelines abstain on the unanswerable question?

**Try it:** replace the questions with your own. Questions that connect two or more topics show the difference best.

This cell makes about 25 model calls and takes two to four minutes.


In [ ]:
QUESTIONS = [
    ('fact', 'What are the total lease liabilities reported by the company?'),
    ('multi-hop', 'How are the suppliers the company depends on connected to the risks it describes for its revenue?'),
    ('multi-hop', 'Which risks relate to key personnel, and which parts of the business would those risks affect?'),
    ('multi-hop', 'How could changes in economic conditions affect both the suppliers of the company and its financial obligations?'),
    ('unanswerable', 'What is the revenue target of AnyCompany for the year 2030?'),
]

def pairwise_judge(question, answer_a, answer_b):
    prompt = ('Question: ' + question + '\n\nAnswer A:\n' + answer_a + '\n\nAnswer B:\n' + answer_b + '\n\n'
              'Which answer is more complete, connects the relevant facts better, and stays grounded in its citations? '
              'If both say the information is not available, that is a TIE. '
              'Return only JSON with the keys winner (A, B or TIE) and reason (one sentence).')
    text, _ = llm(prompt, max_tokens=200)
    match = re.search(r'\{.*\}', text, re.S)
    try:
        return json.loads(match.group(0)) if match else {'winner': 'TIE', 'reason': 'No verdict returned'}
    except json.JSONDecodeError:
        return {'winner': 'TIE', 'reason': 'The verdict could not be read'}

PIPELINES = [('Vector RAG', vector_retrieve), ('GraphRAG', graph_retrieve)]
random.seed(42)
rows = []

for question_type, question in QUESTIONS:
    query_id = uuid.uuid4().hex[:8]
    answers = {}
    print('#' * 80)
    print('QUESTION (' + question_type + '):', question)
    for mode, retrieve_fn in PIPELINES:
        chunks, latency = retrieve_fn(question)
        answer, usage = generate(question, chunks)
        check = check_answer(answer, chunks)
        answers[mode] = answer
        log_event(query_id=query_id, mode=mode, question_type=question_type, question=question,
                  latency_s=round(latency, 3), input_tokens=usage['inputTokens'],
                  chunks=[{'id': c['id'], 'file': c['file'], 'page': c['page'], 'score': c['score']} for c in chunks],
                  answer=answer, risk=check['risk'], issues=check['issues'])
        rows.append({'query_id': query_id, 'type': question_type, 'question': question[:60], 'pipeline': mode,
                     'retrieval_s': round(latency, 2), 'input_tokens': usage['inputTokens'],
                     'pages_covered': len({c['page'] for c in chunks}), 'risk': check['risk']})
        print()
        print('[' + mode + ']', RISK_ICON[check['risk']], check['risk'])
        print(answer)
    order = ['Vector RAG', 'GraphRAG']
    random.shuffle(order)
    verdict = pairwise_judge(question, answers[order[0]], answers[order[1]])
    winner = {'A': order[0], 'B': order[1]}.get(str(verdict.get('winner', '')).upper(), 'TIE')
    print()
    print('Judge winner:', winner, '-', verdict.get('reason', ''))
    for row in rows:
        if row['query_id'] == query_id:
            row['judge_winner'] = winner

comparison = pd.DataFrame(rows).drop(columns=['query_id'])
print()
display(comparison)
print('Wins by question type')
display(comparison.drop_duplicates('question').groupby(['type', 'judge_winner']).size().unstack(fill_value=0))

### 20. Monitoring the Pipeline

When an answer is wrong, the cause is usually the retrieval. The log written in Step 19 shows exactly what each pipeline gave the model.

- The log is read back from `logs/graphrag_log.jsonl` into a table.
- **Average per pipeline**: retrieval latency, input tokens and the number of distinct pages each answer was based on.
- **Risk levels per pipeline**: how many answers were LOW, MEDIUM, HIGH or ABSTAINED.
- **Most retrieved pages**: the pages that both pipelines rely on most. A page that appears for every question can be a sign of a chunk that is too general.

**What to look for:** GraphRAG usually covers more pages and sends more tokens. Decide whether its quality gain on multi-hop questions is worth that cost for your use case.


In [ ]:
with open(LOG_FILE) as f:
    events = [json.loads(line) for line in f if line.strip()]
rag_events = [e for e in events if e.get('mode') in ('Vector RAG', 'GraphRAG')]
log = pd.DataFrame(rag_events)
log['pages_covered'] = log['chunks'].apply(lambda cs: len({c['page'] for c in cs}))

print('Events logged:', len(events))
print()
print('Average per pipeline')
display(log.groupby('mode')[['latency_s', 'input_tokens', 'pages_covered']].mean().round(2))

print('Risk levels per pipeline')
display(log.groupby(['mode', 'risk']).size().unstack(fill_value=0))

pages = [(e['mode'], c['file'], c['page']) for e in rag_events for c in e['chunks']]
print('Most retrieved pages')
display(pd.DataFrame(pages, columns=['pipeline', 'file', 'page'])
        .value_counts().head(10).rename('times_retrieved').reset_index())

# Agentic GraphRAG (Bonus)

### 21. Agentic GraphRAG with Text-to-openCypher

So far the Knowledge Base decided how to use the graph. In **agentic GraphRAG** the model decides: it reads the graph schema, writes its own openCypher query, runs it and fixes it if it fails.

- `GRAPH_SCHEMA`: the labels, relationship types and entity properties found in Steps 10 to 12, given to the model as JSON.
- `WRITE_OPERATIONS`: a safety rule. Any query containing CREATE, MERGE, DELETE, SET, REMOVE or DROP is blocked before it reaches the graph. An agent should never be able to change your data by accident.
- `ask_graph()`, up to three attempts:
  1. the model writes a query,
  2. the query is checked against the safety rule,
  3. the query runs; if it fails, the error message is sent back to the model for the next attempt,
  4. the model turns the result rows into an answer.

This loop of schema, query, error and retry is the same pattern used by the Neptune MCP server and by agent frameworks.

**Try it:** ask your own questions, such as `Which entities are connected to both a risk and a supplier?`


In [ ]:
WRITE_OPERATIONS = re.compile(r'\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP)\b|neptune\.load', re.I)
GRAPH_SCHEMA = json.dumps({
    'node_labels': labels,
    'relationship_types': rels,
    'entity_label': ENTITY_LABEL,
    'entity_properties': entity_keys,
    'entity_name_property': NAME_PROP,
}, default=str)

def clean_query(text):
    text = text.strip()
    if text.startswith('```'):
        text = text.strip('`').strip()
        if text.lower().startswith('cypher'):
            text = text[6:]
    return text.strip()

def ask_graph(question, max_attempts=3):
    error = None
    for attempt in range(1, max_attempts + 1):
        prompt = ('Graph schema (JSON):\n' + GRAPH_SCHEMA + '\n\n'
                  'Write ONE read-only openCypher query for Amazon Neptune Analytics that answers this question: '
                  + question + '\n'
                  'Use only the labels, relationship types and properties in the schema. Put label names in backticks. '
                  'End the query with LIMIT 25. Return only the query, without explanation or code fences.')
        if error:
            prompt += '\n\nYour previous query failed with this error: ' + error + '\nWrite a corrected query.'
        query, _ = llm(prompt, max_tokens=400)
        query = clean_query(query)
        print('Attempt', attempt)
        print(query)
        print()
        if WRITE_OPERATIONS.search(query):
            error = 'Write operations are not allowed. Use MATCH and RETURN only.'
            print('Blocked: the query tries to change data.')
            print()
            continue
        try:
            result_rows = cypher(query)
        except Exception as e:
            error = str(e)[:300]
            print('Query failed:', error)
            print()
            continue
        print('Query returned', len(result_rows), 'rows')
        summary, _ = llm('Question: ' + question + '\n\nGraph query results (JSON):\n'
                         + json.dumps(result_rows, default=str)[:6000]
                         + '\n\nAnswer the question using only these results. If the results are empty, say so.')
        log_event(mode='agentic-cypher', question=question, query=query, rows=len(result_rows), answer=summary)
        return summary
    return 'The agent could not write a valid query in ' + str(max_attempts) + ' attempts.'

agent_answer = ask_graph('Which entities are connected to the most other entities, and what are they connected to?')
print()
print('ANSWER')
print(agent_answer)

### 22. Connecting an AI Assistant through the Neptune MCP Server

The **Model Context Protocol (MCP)** is an open standard that lets AI assistants use external tools. The **AWS Labs Amazon Neptune MCP Server** gives an assistant three abilities on your graph: check its status, read its schema and run openCypher queries.

This cell prints a ready-to-use MCP client configuration with your graph ID filled in.

- `command` and `args`: start the server with `uvx`, which downloads and runs the latest version.
- `NEPTUNE_ENDPOINT`: the graph, in the form `neptune-graph://<graph id>`.
- `AWS_REGION`: the Region of the graph.

**Optional:** to try it, paste the JSON into the MCP settings of a client such as Claude Desktop or Kiro, on a computer that has `uv` installed and AWS credentials with access to the graph. Then ask the assistant questions about your graph in plain English. This step is not needed to complete the lab, and the graph must still exist, so do it before Step 23.


In [ ]:
mcp_config = {
    'mcpServers': {
        'Neptune Query': {
            'command': 'uvx',
            'args': ['awslabs.amazon-neptune-mcp-server@latest'],
            'env': {
                'NEPTUNE_ENDPOINT': 'neptune-graph://' + GRAPH_ID,
                'AWS_REGION': REGION,
                'FASTMCP_LOG_LEVEL': 'ERROR',
            },
        }
    }
}
print(json.dumps(mcp_config, indent=2))

# Delete and Cleanup

### 23. Deleting All Lab Resources

This cell deletes everything the notebook created, using the IDs saved in `lab_state.json`. It works even after a kernel restart: run Steps 1 and 2, then this cell.

The order matters:

1. **Data source**, then **Knowledge Base**. `wait_until_gone()` waits until each one really disappears.
2. **Neptune Analytics graph**, with `skipSnapshot=True` so no backup is kept. **Deleting the Knowledge Base does not delete the graph**, and the graph is the expensive part.
3. **IAM role**: the inline policy first, then the role.
4. **S3 bucket**: all objects first, because S3 does not delete a bucket that still has objects in it.

Each step catches its own errors, so one problem does not stop the rest of the cleanup.


In [ ]:
def wait_until_gone(get_fn, label, every=10, timeout=900):
    start = time.time()
    while time.time() - start < timeout:
        try:
            get_fn()
        except ClientError as e:
            if e.response['Error']['Code'] == 'ResourceNotFoundException':
                print(label, 'deleted')
                return
            raise
        time.sleep(every)
    print(label, 'is still deleting. Check again in a few minutes.')

st = json.load(open(STATE_FILE)) if os.path.exists(STATE_FILE) else {}

# 1. Data source and Knowledge Base (must be deleted before the graph)
if st.get('kb_id'):
    if st.get('ds_id'):
        try:
            bedrock_agent.delete_data_source(knowledgeBaseId=st['kb_id'], dataSourceId=st['ds_id'])
            wait_until_gone(lambda: bedrock_agent.get_data_source(knowledgeBaseId=st['kb_id'],
                                                                  dataSourceId=st['ds_id']), 'Data source')
        except ClientError as e:
            print('Data source:', e.response['Error']['Code'])
    try:
        bedrock_agent.delete_knowledge_base(knowledgeBaseId=st['kb_id'])
        wait_until_gone(lambda: bedrock_agent.get_knowledge_base(knowledgeBaseId=st['kb_id']), 'Knowledge Base')
    except ClientError as e:
        print('Knowledge Base:', e.response['Error']['Code'])

# 2. Neptune Analytics graph (deleting the Knowledge Base does not delete it)
if st.get('graph_id'):
    try:
        neptune.delete_graph(graphIdentifier=st['graph_id'], skipSnapshot=True)
        print('Graph deletion started:', st['graph_id'])
    except ClientError as e:
        print('Graph:', e.response['Error']['Message'])

# 3. IAM role
if st.get('role_name'):
    try:
        iam.delete_role_policy(RoleName=st['role_name'], PolicyName=POLICY_NAME)
    except ClientError as e:
        print('Role policy:', e.response['Error']['Code'])
    try:
        iam.delete_role(RoleName=st['role_name'])
        print('IAM role deleted')
    except ClientError as e:
        print('IAM role:', e.response['Error']['Code'])

# 4. S3 bucket and its contents
if st.get('bucket'):
    try:
        for page in s3.get_paginator('list_objects_v2').paginate(Bucket=st['bucket']):
            objects = [{'Key': o['Key']} for o in page.get('Contents', [])]
            if objects:
                s3.delete_objects(Bucket=st['bucket'], Delete={'Objects': objects})
        s3.delete_bucket(Bucket=st['bucket'])
        print('S3 bucket deleted')
    except ClientError as e:
        print('S3 bucket:', e.response['Error']['Code'])

### 24. Verifying That Nothing Is Left Running

- The cell waits one minute, then lists every Neptune Analytics graph whose name starts with `k21-graphrag-`.
- A graph in status `DELETING` is fine. Run the cell again after a few minutes until it shows `None`.
- A graph in status `AVAILABLE` means the deletion did not start. Run Step 23 again.
- When nothing is left, the state file is renamed to `lab_state_deleted.json`, so running the notebook again starts a fresh lab instead of reusing deleted IDs.

**After this cell:** stop and delete the JupyterLab space, the user profile and the SageMaker domain, as described in section 9.2 of the lab guide.


In [ ]:
print('Waiting 60 seconds before checking...')
time.sleep(60)
remaining = [g for g in neptune.list_graphs()['graphs'] if g['name'].startswith('k21-graphrag-')]
if remaining:
    for g in remaining:
        print('Still present:', g['name'], g['status'])
    print('Run this cell again in a few minutes. If a graph shows AVAILABLE, run Step 23 again.')
else:
    print('Remaining lab graphs: None')
    if os.path.exists(STATE_FILE):
        os.replace(STATE_FILE, 'lab_state_deleted.json')
    print('All lab resources are deleted. Now delete the SageMaker space and domain (lab guide section 9.2).')

### Conclusion

In this lab, we learned the following key concepts and techniques:

1. **Building a knowledge graph from documents**
   - We created an Amazon Neptune Analytics graph and a GraphRAG Knowledge Base entirely from code.
   - We used a graph construction model to extract entities and relationships from every chunk during ingestion.

2. **Exploring a graph database**
   - We queried the graph with openCypher to see its labels, relationships, hubs and neighbourhoods.
   - We ranked entities with degree and PageRank, computed inside the graph.

3. **GraphRAG retrieval**
   - We retrieved graph-enhanced context with the Retrieve API and built a vector-only baseline to compare against.

4. **Trustworthy answers**
   - We generated answers with numbered citations and rated each one with citation, number and LLM-judge checks.

5. **Measuring when GraphRAG helps**
   - We compared both pipelines on fact, multi-hop and unanswerable questions, and logged latency, tokens, pages and risk.

6. **Agentic GraphRAG**
   - We built an agent that writes, checks and retries its own read-only openCypher queries, and generated an MCP configuration for the Neptune MCP server.

By the end of this lab, we gained hands-on experience with every layer of a GraphRAG system, from the graph database to the evaluated answer, and the evidence to decide when a knowledge graph is worth adding to a RAG application.
